In [1]:
import re

def clean_html_with_regex(file_path):
    # Read file
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove <script>...</script> and <style>...</style> blocks
    text = re.sub(r'<(style).*?>.*?</\1>', '', text, flags=re.DOTALL | re.IGNORECASE)

    # Remove ALL remaining HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Convert HTML entities (like &amp;, &#128545;) into plain text
    text = re.sub(r'&[a-zA-Z#0-9]+;', ' ', text)

    # Remove multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [2]:
file_path = "nlp_preprocessing_1.txt"  # 🔹 your HTML or .txt file
cleaned_text = clean_html_with_regex(file_path)

print("Cleaned Text:\n")
print(cleaned_text[:1000])  # print first 1000 characters


Cleaned Text:

Just got the new Apple Vision Pro, and it's 🤯! The spatial computing is mind-blowing. But I've been reading a lot about data privacy issues. The article at https://www.theverge.com talks about it. Customer reviews on their site are mixed. One user wrote, "The AR features were a game-changer!!!" but another said, "I have so many bugs... it's a huge waste of $$$." The data looks so messy with all the exclamation marks, periods, and the dollar signs. User Feedback "This is the worst product I've ever owned. Total disappointment." "The battery life is amazing. Definitely a must-buy for tech enthusiasts." "It's revolutionary. A new era of computing." Some users are still confused about the setup process need better documentation. console.log("This is some JavaScript code that should be removed."); var user = "customer"; // Don't process this line! The sentiment analysis needs to be accurate. Words like "waste" and "disappointment" should be classified as negative. While "revo

In [3]:
with open("cleaned_output.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("💾 Cleaned text saved to cleaned_output.txt")


💾 Cleaned text saved to cleaned_output.txt


In [4]:
!pip install nltk



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')



[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\SAIKRISHNA\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [6]:
file_path = "cleaned_output.txt"  # path to your cleaned text file


In [7]:
# Read the cleaned text file
file_path = "cleaned_output.txt"  # change name if needed

with open(file_path, "r", encoding="utf-8") as f:
    cleaned_text = f.read()

print("✅ File loaded successfully!")
print("Preview of text:\n", cleaned_text[:1000])


✅ File loaded successfully!
Preview of text:
 Just got the new Apple Vision Pro, and it's 🤯! The spatial computing is mind-blowing. But I've been reading a lot about data privacy issues. The article at https://www.theverge.com talks about it. Customer reviews on their site are mixed. One user wrote, "The AR features were a game-changer!!!" but another said, "I have so many bugs... it's a huge waste of $$$." The data looks so messy with all the exclamation marks, periods, and the dollar signs. User Feedback "This is the worst product I've ever owned. Total disappointment." "The battery life is amazing. Definitely a must-buy for tech enthusiasts." "It's revolutionary. A new era of computing." Some users are still confused about the setup process need better documentation. console.log("This is some JavaScript code that should be removed."); var user = "customer"; // Don't process this line! The sentiment analysis needs to be accurate. Words like "waste" and "disappointment" should be clas

In [8]:
# Initialize the sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Split text into sentences (basic split by '.')
sentences = cleaned_text.split('.')

# Analyze each sentence
results = []
for s in sentences:
    if len(s.strip()) > 5:  # ignore very short sentences
        sentiment = sia.polarity_scores(s)
        compound = sentiment['compound']
        label = 'Positive' if compound > 0.05 else 'Negative' if compound < -0.05 else 'Neutral'
        results.append((s.strip(), label, compound))

# Display results
for sent, label, score in results[:10]:  # show first 10
    print(f"[{label}] ({score}): {sent}")


[Positive] (0.3164): Just got the new Apple Vision Pro, and it's 🤯! The spatial computing is mind-blowing
[Neutral] (0.0): But I've been reading a lot about data privacy issues
[Neutral] (0.0): The article at https://www
[Neutral] (0.0): theverge
[Neutral] (0.0): com talks about it
[Neutral] (0.0): Customer reviews on their site are mixed
[Neutral] (0.0): One user wrote, "The AR features were a game-changer!!!" but another said, "I have so many bugs
[Negative] (-0.128): it's a huge waste of $$$
[Negative] (-0.5009): " The data looks so messy with all the exclamation marks, periods, and the dollar signs
[Negative] (-0.6249): User Feedback "This is the worst product I've ever owned


In [9]:
import pandas as pd

df = pd.DataFrame(results, columns=['Sentence', 'Sentiment', 'Score'])
df.to_csv("sentiment_results.csv", index=False, encoding="utf-8")

print("💾 Sentiment results saved to sentiment_results.csv")


💾 Sentiment results saved to sentiment_results.csv


In [10]:
!pip install scikit-learn numpy pandas



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


In [12]:
# Load your cleaned text
with open("cleaned_output.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Split into documents (e.g., by paragraph or period)
docs = [t.strip() for t in text.split('.') if len(t.strip()) > 30]


In [13]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(docs)


In [14]:
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X)


,n_components,3
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [15]:
words = vectorizer.get_feature_names_out()

for idx, topic in enumerate(lda.components_):
    print(f"\n🧩 Topic {idx+1}:")
    print([words[i] for i in topic.argsort()[-10:][::-1]])  # top 10 words



🧩 Topic 1:
['like', 'data', 'just', 'apple', 'tech', 'mind', 'blowing', 'hashtags', 'noise', 'usernames']

🧩 Topic 2:
['disappointment', 'words', 'need', 'ar', 'said', 'wrote', 'bugs', 'features', 'game', 'changer']

🧩 Topic 3:
['user', 'process', 'customer', 've', 'var', 'line', 'needs', 'don', 'analysis', 'accurate']


In [16]:
!pip install pyLDAvis
import pyLDAvis.sklearn
pyLDAvis.enable_notebook()

pyLDAvis.sklearn.prepare(lda, X, vectorizer)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.6 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.6 MB 1.6 MB/s eta 0:00:02
   -------------------- ------------------- 1.3/2.6 MB 1.8 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.6 MB 1.8 MB/s eta 0:00:01
   ------------------------------------ --- 2.4/2.6 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 2.2 MB/s  0:00:01
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.6/24.4 MB 2.8 MB/s eta 0:00:09
   --- ------------------------------------ 2.1/24.4 MB 2.6 MB/s eta 0:00:09
   ---- ----------------------------------- 2

ModuleNotFoundError: No module named 'pyLDAvis.sklearn'